# 📖 Notebook 2: Analyzers and Mappings

In Notebook 1, we let Elasticsearch automatically detect our field types (dynamic mapping).
Now we'll take control and learn **how** Elasticsearch processes text and **why** mappings matter.

## Learning Objectives

By the end of this notebook, you'll understand:
- How Elasticsearch breaks text into searchable tokens (analysis)
- The difference between `text` and `keyword` field types
- How to create custom analyzers for better search results
- How to design mappings that balance search quality and performance

## 🛠️ Setup

```bash
cd deep-dives/elasticsearch
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from elasticsearch import Elasticsearch
import json

es = Elasticsearch("http://localhost:9200")
info = es.info()
print(f"✅ Connected to Elasticsearch {info['version']['number']}")

## 🔬 What Is Analysis?

When you index a document, Elasticsearch doesn't store the text as-is.
It runs the text through an **analyzer** which breaks it into **tokens**.

An analyzer has three stages:

```
Raw Text → [Character Filters] → [Tokenizer] → [Token Filters] → Tokens
```

| Stage | What It Does | Example |
|-------|-------------|--------|
| Character Filter | Cleans up raw text | Strip HTML tags: `<b>hello</b>` → `hello` |
| Tokenizer | Splits text into tokens | `"The Great Gatsby"` → `["The", "Great", "Gatsby"]` |
| Token Filter | Transforms each token | Lowercase: `["The", "Great"]` → `["the", "great"]` |

Let's see this in action using the **Analyze API** — a tool that shows exactly
how Elasticsearch processes any text.

In [ ]:
def show_analysis(analyzer, text, index=None):
    """Show how an analyzer breaks text into tokens."""
    params = {"analyzer": analyzer, "text": text}
    if index:
        result = es.indices.analyze(index=index, body=params)
    else:
        result = es.indices.analyze(body=params)

    tokens = [t["token"] for t in result["tokens"]]
    print(f"Analyzer:  {analyzer}")
    print(f"Input:     \"{text}\"")
    print(f"Tokens:    {tokens}")
    print()

In [ ]:
# The standard analyzer: lowercase + split on whitespace and punctuation
text = "The Great Gatsby: A Novel by F. Scott Fitzgerald (1925)"

print("🔬 Comparing Built-in Analyzers")
print("=" * 60)
print()

show_analysis("standard", text)
show_analysis("simple", text)
show_analysis("whitespace", text)
show_analysis("keyword", text)

### What Each Analyzer Does

| Analyzer | Tokenizer | Token Filters | Best For |
|----------|----------|--------------|----------|
| `standard` | Splits on word boundaries | Lowercase | General text (default) |
| `simple` | Splits on non-letters | Lowercase | Simple text without numbers |
| `whitespace` | Splits on whitespace only | None | When you want to preserve case and punctuation |
| `keyword` | No splitting (entire input = 1 token) | None | Exact match fields (IDs, emails, status codes) |

💡 The **standard** analyzer is the default. It's good for most text, but
sometimes you need more control.

## 📝 Text vs Keyword: The Most Important Distinction

Elasticsearch has two main string field types:

| Type | Analyzed? | Use Case | Example |
|------|----------|----------|--------|
| `text` | ✅ Yes | Full-text search | Book titles, descriptions, blog content |
| `keyword` | ❌ No | Exact match, sorting, aggregations | IDs, emails, status, categories |

This is the **most common mistake** beginners make: using `text` for a field
that should be `keyword`, or vice versa.

Let's see the difference!

In [ ]:
# Create an index with explicit text and keyword fields
INDEX_NAME = "analyzer_demo"

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "title_text": {"type": "text"},              # analyzed for full-text search
            "title_keyword": {"type": "keyword"},         # stored as-is for exact match
            "author": {"type": "text"},
            "status": {"type": "keyword"},                # exact match only
            "tags": {"type": "keyword"},                  # exact match for filtering
            "description": {"type": "text"}
        }
    },
    settings={"number_of_shards": 1, "number_of_replicas": 0}
)

# Index a document with the same title in both fields
es.index(index=INDEX_NAME, id=1, document={
    "title_text": "The Great Gatsby",
    "title_keyword": "The Great Gatsby",
    "author": "F. Scott Fitzgerald",
    "status": "published",
    "tags": ["classic", "fiction"],
    "description": "A novel about the American Dream"
})
es.indices.refresh(index=INDEX_NAME)

print("✅ Created index with text and keyword fields")

In [ ]:
# TEXT field: searching for "great" matches because the text was tokenized
result = es.search(index=INDEX_NAME, query={"match": {"title_text": "great"}})
text_hits = result["hits"]["total"]["value"]

# KEYWORD field: searching for "great" does NOT match — it needs the exact string
result = es.search(index=INDEX_NAME, query={"match": {"title_keyword": "great"}})
keyword_hits = result["hits"]["total"]["value"]

# KEYWORD field: searching for the EXACT string matches
result = es.search(index=INDEX_NAME, query={"match": {"title_keyword": "The Great Gatsby"}})
exact_hits = result["hits"]["total"]["value"]

print("📊 Text vs Keyword Search Results")
print("=" * 50)
print(f"  Search 'great' in title_text (text):       {text_hits} hit(s) ✅")
print(f"  Search 'great' in title_keyword (keyword):  {keyword_hits} hit(s) ❌")
print(f"  Search exact 'The Great Gatsby' (keyword):  {exact_hits} hit(s) ✅")
print()
print("💡 'text' fields are analyzed (broken into tokens) → partial matches work.")
print("   'keyword' fields are stored as-is → only exact matches work.")

## 🎨 Custom Analyzers

The built-in analyzers are great, but sometimes you need more control.
For example:

- **Stop words**: Remove common words like "the", "a", "is" that add noise
- **Stemming**: Reduce words to their root — "running" → "run", "books" → "book"
- **Synonyms**: Make "laptop" also match "notebook computer"

Let's build a custom analyzer step by step!

In [ ]:
# Create an index with a custom analyzer
CUSTOM_INDEX = "custom_analyzer_demo"

if es.indices.exists(index=CUSTOM_INDEX):
    es.indices.delete(index=CUSTOM_INDEX)

es.indices.create(
    index=CUSTOM_INDEX,
    settings={
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "analysis": {
            "analyzer": {
                "my_english_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": [
                        "lowercase",        # THE → the
                        "english_stop",      # remove "the", "a", "is"
                        "english_stemmer"    # running → run
                    ]
                }
            },
            "filter": {
                "english_stop": {
                    "type": "stop",
                    "stopwords": "_english_"
                },
                "english_stemmer": {
                    "type": "stemmer",
                    "language": "english"
                }
            }
        }
    },
    mappings={
        "properties": {
            "title": {
                "type": "text",
                "analyzer": "my_english_analyzer"  # use our custom analyzer
            },
            "description": {
                "type": "text",
                "analyzer": "my_english_analyzer"
            }
        }
    }
)

print("✅ Created index with custom English analyzer")

In [ ]:
# Compare standard vs our custom analyzer
text = "The runners were running quickly through the beautiful gardens"

print("🔬 Standard vs Custom Analyzer")
print("=" * 60)
print()

show_analysis("standard", text)
show_analysis("my_english_analyzer", text, index=CUSTOM_INDEX)

print("💡 Notice the custom analyzer:")
print("   - Removed stop words: 'the', 'were', 'through'")
print("   - Stemmed words: 'runners' → 'runner', 'running' → 'run'")
print("   - 'quickly' → 'quick', 'beautiful' → 'beauti', 'gardens' → 'garden'")

In [ ]:
# Let's see why stemming matters for search quality
books = [
    {"title": "Running with Scissors", "description": "A memoir about a chaotic childhood"},
    {"title": "Born to Run", "description": "The hidden tribe of superathletes and runners"},
    {"title": "The Runner's World", "description": "A guide for beginning and experienced runners"},
]

for i, book in enumerate(books):
    es.index(index=CUSTOM_INDEX, id=i+1, document=book)
es.indices.refresh(index=CUSTOM_INDEX)

# Search for "running" — with stemming, this also matches "run" and "runners"
print("🔍 Search for 'running' with stemming analyzer")
print("=" * 50)

results = es.search(
    index=CUSTOM_INDEX,
    query={"match": {"description": "running"}}
)

for hit in results["hits"]["hits"]:
    src = hit["_source"]
    print(f"  📗 {src['title']} (score: {hit['_score']:.4f})")
    print(f"     {src['description']}")
    print()

print("💡 Stemming turned 'running' → 'run', so it matched 'runners' and 'run' too!")

### Synonyms: Making "laptop" match "notebook computer"

Search engines need to understand that different words can mean the same thing.
A user searching for **"tv"** probably also wants to see results for
**"television"**. Elasticsearch handles this with the **synonym** token filter.

Synonym rules can be:
- **Equivalent** — `tv, television, telly` (all map to each other)
- **Explicit/one-way** — `laptop => notebook computer` (queries for "laptop" also find "notebook computer" but not vice versa)


In [ ]:
# Custom analyzer with a synonym filter
SYNONYM_INDEX = "synonym_demo"

if es.indices.exists(index=SYNONYM_INDEX):
    es.indices.delete(index=SYNONYM_INDEX)

es.indices.create(
    index=SYNONYM_INDEX,
    settings={
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "analysis": {
            "filter": {
                "my_synonyms": {
                    "type": "synonym",
                    "synonyms": [
                        "tv, television, telly",
                        "laptop, notebook computer",
                        "phone, mobile, cellphone, smartphone"
                    ]
                }
            },
            "analyzer": {
                "synonym_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "my_synonyms"]
                }
            }
        }
    },
    mappings={"properties": {"name": {"type": "text", "analyzer": "synonym_analyzer"}}}
)

# Index products using a variety of wordings
for i, name in enumerate([
    "55-inch OLED television",
    "Smart TV 4K",
    "Gaming laptop",
    "Ultralight notebook computer",
    "Wireless smartphone charger",
]):
    es.index(index=SYNONYM_INDEX, id=i+1, document={"name": name})
es.indices.refresh(index=SYNONYM_INDEX)

# Search for "tv" — should find both "television" and "TV" products
print("🔍 Search for 'tv' (will also match 'television' via synonyms)")
r = es.search(index=SYNONYM_INDEX, query={"match": {"name": "tv"}})
for hit in r["hits"]["hits"]:
    print(f"  📺 {hit['_source']['name']}  (score: {hit['_score']:.3f})")

print("\n🔍 Search for 'laptop' (will also match 'notebook computer')")
r = es.search(index=SYNONYM_INDEX, query={"match": {"name": "laptop"}})
for hit in r["hits"]["hits"]:
    print(f"  💻 {hit['_source']['name']}  (score: {hit['_score']:.3f})")

print("\n💡 Synonyms are resolved at analysis time, so they apply to both")
print("   indexed documents AND search queries. Powerful for product search!")


## 🗺️ Mapping Design Best Practices

A **mapping** is the schema of your index. It tells Elasticsearch:
- What fields exist in your documents
- What type each field is (text, keyword, integer, date, etc.)
- How each field should be analyzed

### Choosing the Right Field Type

| If you want to... | Use this type |
|-------------------|---------------|
| Search within text ("find books about dogs") | `text` |
| Filter by exact value (status = "published") | `keyword` |
| Sort by or aggregate a number | `integer`, `float`, `long` |
| Filter by date range | `date` |
| Search AND filter the same string | `text` + `keyword` (multi-field) |

### Multi-Fields: The Best of Both Worlds

Sometimes you need both full-text search AND exact matching on the same field.
Elasticsearch supports this with **multi-fields**.

In [ ]:
# Create a well-designed bookstore mapping
BOOKSTORE_INDEX = "bookstore"

if es.indices.exists(index=BOOKSTORE_INDEX):
    es.indices.delete(index=BOOKSTORE_INDEX)

es.indices.create(
    index=BOOKSTORE_INDEX,
    settings={
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "analysis": {
            "analyzer": {
                "english_text": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "english_stop", "english_stemmer"]
                }
            },
            "filter": {
                "english_stop": {"type": "stop", "stopwords": "_english_"},
                "english_stemmer": {"type": "stemmer", "language": "english"}
            }
        }
    },
    mappings={
        "properties": {
            # Multi-field: search by title (text) OR sort/aggregate by title (keyword)
            "title": {
                "type": "text",
                "analyzer": "english_text",
                "fields": {
                    "raw": {"type": "keyword"}  # access as title.raw
                }
            },
            # Author: search by name, but also allow exact filtering
            "author": {
                "type": "text",
                "fields": {
                    "raw": {"type": "keyword"}
                }
            },
            "description": {"type": "text", "analyzer": "english_text"},
            "price": {"type": "float"},
            "publish_date": {"type": "date"},
            "categories": {"type": "keyword"},   # exact match for filtering/aggregation
            "rating": {"type": "float"},
            "in_stock": {"type": "boolean"},
            "isbn": {"type": "keyword"}           # always exact match
        }
    }
)

print("✅ Created 'bookstore' index with optimized mapping")
print()
print("📋 Mapping design decisions:")
print("  title:        text (search) + keyword sub-field (sort/aggregate)")
print("  author:       text (search) + keyword sub-field (filter by exact name)")
print("  description:  text only (search, never need to sort by it)")
print("  price:        float (range filters, sorting)")
print("  categories:   keyword (exact match filters, aggregations)")
print("  isbn:         keyword (always exact lookup)")

In [ ]:
# Index some books into our well-designed index
books = [
    {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald", "description": "A novel about the American Dream in the Jazz Age", "price": 9.99, "publish_date": "1925-04-10", "categories": ["Classic", "Fiction"], "rating": 4.5, "in_stock": True, "isbn": "978-0743273565"},
    {"title": "Great Expectations", "author": "Charles Dickens", "description": "A coming-of-age story about ambition and love in Victorian England", "price": 7.99, "publish_date": "1861-08-01", "categories": ["Classic", "Fiction"], "rating": 4.2, "in_stock": True, "isbn": "978-0141439563"},
    {"title": "Clean Code", "author": "Robert C. Martin", "description": "A handbook of agile software craftsmanship for writing better code", "price": 29.99, "publish_date": "2008-08-01", "categories": ["Technology", "Programming"], "rating": 4.3, "in_stock": True, "isbn": "978-0132350884"},
]

for i, book in enumerate(books):
    es.index(index=BOOKSTORE_INDEX, id=i+1, document=book)
es.indices.refresh(index=BOOKSTORE_INDEX)

# Demonstrate multi-field: search title (text) vs sort by title.raw (keyword)
print("🔍 Multi-field demo: title (text) vs title.raw (keyword)")
print("=" * 60)

# Search works on the text field
results = es.search(
    index=BOOKSTORE_INDEX,
    query={"match": {"title": "great"}}
)
print(f"\nSearch 'great' in title (text): {results['hits']['total']['value']} hits")
for hit in results["hits"]["hits"]:
    print(f"  📗 {hit['_source']['title']}")

# Sorting works on the keyword sub-field
results = es.search(
    index=BOOKSTORE_INDEX,
    query={"match_all": {}},
    sort=[{"title.raw": "asc"}]  # sort alphabetically by exact title
)
print(f"\nAll books sorted by title.raw (keyword):")
for hit in results["hits"]["hits"]:
    print(f"  📗 {hit['_source']['title']}")

## 📏 Viewing Your Mapping

You can always check what mapping Elasticsearch is using for an index.
This is helpful for debugging search issues — maybe a field is `keyword`
when you expected `text`!

In [ ]:
# View the mapping for our bookstore index
mapping = es.indices.get_mapping(index=BOOKSTORE_INDEX)

print("📋 Current mapping for 'bookstore' index:")
print(json.dumps(mapping.body[BOOKSTORE_INDEX]["mappings"]["properties"], indent=2))

## ⚠️ Mapping Pitfalls

### Pitfall 1: Mapping Explosion

If you index documents with many dynamic fields, Elasticsearch creates a mapping
entry for **every** field. This wastes memory and slows searches.

**Fix**: Define your mapping explicitly and set `dynamic: false` or `dynamic: strict`.

### Pitfall 2: You Can't Change a Field's Type

Once a field is mapped, you **cannot** change its type. If `title` is `text`,
you can't change it to `keyword` later. You'd need to create a new index
and reindex your data.

**Fix**: Design your mappings carefully before indexing data.

### Pitfall 3: Not All Fields Need to Be Searchable

If you have 20 fields but only search by 3, you're wasting resources indexing
the other 17. Use `"enabled": false` for fields you only need in `_source`.

In [ ]:
# Demonstrate strict mapping: reject unknown fields
STRICT_INDEX = "strict_demo"

if es.indices.exists(index=STRICT_INDEX):
    es.indices.delete(index=STRICT_INDEX)

es.indices.create(
    index=STRICT_INDEX,
    settings={"number_of_shards": 1, "number_of_replicas": 0},
    mappings={
        "dynamic": "strict",  # reject documents with unmapped fields
        "properties": {
            "title": {"type": "text"},
            "price": {"type": "float"}
        }
    }
)

# This works — title and price are in the mapping
es.index(index=STRICT_INDEX, id=1, document={"title": "Valid Book", "price": 9.99})
print("✅ Indexed document with known fields")

# This fails — 'author' is not in the mapping
try:
    es.index(index=STRICT_INDEX, id=2, document={"title": "Bad Book", "price": 9.99, "author": "Unknown"})
except Exception as e:
    print(f"❌ Rejected: {str(e)[:120]}...")
    print("\n💡 Strict mapping prevents 'mapping explosion' by rejecting unknown fields.")

## 🧪 Exercises

1. **Analyze your own text**: Use `show_analysis()` with different analyzers on a sentence of your choice
2. **Build a synonym analyzer**: Create a custom analyzer that treats "laptop" and "notebook" as the same word (hint: use a `synonym` token filter)
3. **Design a mapping for a movie database**: Think about which fields should be `text`, `keyword`, `integer`, `date`, etc.
4. **Compare search results**: Index the same data with standard vs custom analyzers and see how search results differ

In [ ]:
# Exercise space — try your experiments here!


## 🎯 Key Takeaways

1. **Analyzers** break text into searchable tokens using: character filters → tokenizer → token filters
2. **`text`** fields are analyzed (tokenized) — use for full-text search
3. **`keyword`** fields are stored as-is — use for exact match, sorting, aggregations
4. **Multi-fields** give you both: search on `title` (text), sort by `title.raw` (keyword)
5. **Custom analyzers** let you add stop words, stemming, synonyms for better search quality
6. **Mappings are immutable** — design them carefully before indexing data
7. Only index fields you actually need to search — extra fields waste memory

**Next up**: Notebook 3 covers **Aggregations and Faceted Search** — how to
build filter panels and compute statistics like an e-commerce site.

## 🧹 Cleanup

Run this cell to delete the demo indices created in this notebook.

In [ ]:
for idx in [INDEX_NAME, CUSTOM_INDEX, BOOKSTORE_INDEX, STRICT_INDEX, SYNONYM_INDEX]:
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)
        print(f"🗑️  Deleted '{idx}'")
print("✅ Cleanup complete")